# 04. Robot Perception — Sensor Models

Sensor model은 특정 pose에서 관측 $z$가 나올 가능성을 계산한다.

$$p(z_t\mid x_t,m)$$

Localization과 SLAM에서 sensor model은 particle weight, EKF innovation, occupancy update를 결정한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

## 1. Range Sensor Likelihood Field

지도 위 장애물까지의 거리장을 만들고, 측정 endpoint가 장애물에 가까울수록 likelihood를 높게 준다.

In [ ]:
np.random.seed(5)
size=80
occ=np.zeros((size,size),dtype=bool)
occ[18:62,20]=True; occ[18:62,60]=True; occ[18,20:61]=True; occ[62,20:61]=True
occ[38:44,36:48]=True
obs=np.argwhere(occ)
Y,X=np.indices((size,size))
dist=np.full((size,size),np.inf)
for oy,ox in obs:
    dist=np.minimum(dist, np.sqrt((X-ox)**2+(Y-oy)**2))

pose=np.array([30.0,30.0,np.deg2rad(20)])
angles=np.deg2rad(np.linspace(-70,70,15))
z_max=38
sigma=2.5

def ray_cast(pose, angle):
    th=pose[2]+angle
    for r in np.linspace(0,z_max,160):
        x=int(round(pose[0]+r*np.cos(th))); y=int(round(pose[1]+r*np.sin(th)))
        if x<0 or x>=size or y<0 or y>=size or occ[y,x]:
            return r
    return z_max

zs=np.array([ray_cast(pose,a)+np.random.randn()*1.0 for a in angles])
endpoints=[]; likelihoods=[]
for a,z in zip(angles,zs):
    ex=pose[0]+z*np.cos(pose[2]+a); ey=pose[1]+z*np.sin(pose[2]+a)
    endpoints.append([ex,ey])
    ix=int(np.clip(round(ex),0,size-1)); iy=int(np.clip(round(ey),0,size-1))
    likelihoods.append(np.exp(-0.5*(dist[iy,ix]/sigma)**2))
endpoints=np.array(endpoints); likelihoods=np.array(likelihoods)

fig, axes=plt.subplots(1,2,figsize=(13,6))
axes[0].imshow(occ,cmap='gray_r',origin='lower')
axes[0].scatter(pose[0],pose[1],color='#E85D24',s=80,label='robot')
for ep in endpoints:
    axes[0].plot([pose[0],ep[0]],[pose[1],ep[1]],color='#534AB7',alpha=0.45)
axes[0].scatter(endpoints[:,0],endpoints[:,1],c=likelihoods,cmap='viridis',s=60,label='beam endpoints')
axes[0].legend(); axes[0].set_title('range endpoints')
axes[1].imshow(dist,cmap='magma_r',origin='lower')
axes[1].scatter(endpoints[:,0],endpoints[:,1],c=likelihoods,cmap='viridis',s=60)
axes[1].set_title('likelihood field: obstacle distance')
plt.tight_layout(); plt.savefig('assets/04_likelihood_field.png',dpi=150,bbox_inches='tight'); plt.show()
print('mean beam likelihood:', likelihoods.mean().round(4))

## 2. Mixture Sensor Model 직관

레이저 모델은 보통 여러 현상을 섞어서 표현한다.

- hit: 예상 거리 근처의 Gaussian
- short: 예상보다 짧은 동적 장애물
- max: 최대거리 반환
- random: 랜덤 측정

In [ ]:
z_star=20.0
z=np.linspace(0,40,500)
p_hit=np.exp(-0.5*((z-z_star)/2.0)**2); p_hit/=np.trapezoid(p_hit,z)
p_short=np.where(z<=z_star, 0.18*np.exp(-0.18*z), 0); p_short/=np.trapezoid(p_short,z)
p_max=np.exp(-0.5*((z-40)/0.35)**2); p_max/=np.trapezoid(p_max,z)
p_rand=np.ones_like(z)/40
mix=0.72*p_hit+0.12*p_short+0.08*p_max+0.08*p_rand
fig, ax=plt.subplots(figsize=(8,5))
ax.plot(z,mix,color='#E85D24',lw=2.5,label='mixture')
ax.plot(z,p_hit,'--',label='hit'); ax.plot(z,p_short,'--',label='short')
ax.plot(z,p_max,'--',label='max'); ax.plot(z,p_rand,'--',label='random')
ax.axvline(z_star,color='gray',ls=':',label='expected')
ax.grid(alpha=0.25); ax.legend(); ax.set_xlabel('range z'); ax.set_ylabel('probability density')
ax.set_title('Beam range finder mixture model')
plt.savefig('assets/04_beam_mixture_model.png',dpi=150,bbox_inches='tight'); plt.show()

## 요약

| 모델 | 의미 | 책 커리큘럼 연결 |
|------|------|------------------|
| Beam model | ray casting 기반 거리 likelihood | Ch.6 Robot Perception |
| Likelihood field | endpoint와 장애물 거리 기반 | MCL에서 계산량 절약 |
| Mixture model | hit/short/max/random 현상 결합 | 실제 센서 노이즈 모델링 |